# Introduction

This notebook shows the effect of Newton-Raphson method for state estimation. We create a small network with two power sensors to show how Newton-Raphson method can split the uncertainties of active and reactive power in the sensors.

# Example network

The network used in the notebook is shown below.

```
source_1-------- node_0 -------------------------line_2-------- node_3 -----------sym_load_4
                  |                     |                                   |
           voltage_sensor_5         power_sensor_6                       power_sensor_7
           u_measured = 10.5e3      p_measured = 1e6                     p_measured = -1e6
           u_sigma = 100.0          q_measured = 1e6                     q_measured = -1e6
                                    p_sigma = 1e4                        p_sigma = 1e9
                                    q_sigma = 1e9                        q_sigma = 1e4
```

The network has two power sensors which have completely opposity measured value. However, the accuracy (error range in sigma) for both sensors are opposite for active and reactive power. So in theory, we should trust the `power_sensor_6` for `p_measured` and `power_sensor_7` for `q_measured`.

# Imports

In [3]:
import pandas as pd
import numpy as np

from power_grid_model import PowerGridModel, LoadGenType, MeasuredTerminalType, initialize_array


# Construct network

In [9]:
# node
node = initialize_array("input", "node", 2)
node["id"] = np.array([0, 3])
node["u_rated"] = [10.5e3, 10.5e3]

# line
line = initialize_array("input", "line", 1)
line["id"] = [2]
line["from_node"] = [0]
line["to_node"] = [3]
line["from_status"] = [1]
line["to_status"] = [1]
line["r1"] = [0.1]
line["x1"] = [0.02]
line["c1"] = [0.0]
line["tan1"] = [0.0]
line["i_n"] = [1000.0]

# load
sym_load = initialize_array("input", "sym_load", 1)
sym_load["id"] = [4]
sym_load["node"] = [3]
sym_load["status"] = [1]
sym_load["type"] = [LoadGenType.const_power]
sym_load["p_specified"] = [1e6]
sym_load["q_specified"] = [-1e6]

# source
source = initialize_array("input", "source", 1)
source["id"] = [1]
source["node"] = [1]
source["status"] = [1]
source["u_ref"] = [1.0]

# voltage sensor
voltage_sensor = initialize_array("input", "sym_voltage_sensor", 1)
voltage_sensor["id"] = 5
voltage_sensor["measured_object"] = 0
voltage_sensor["u_sigma"] = [100.0]
voltage_sensor["u_measured"] = [10.5e3]

# power sensor
power_sensor = initialize_array("input", "sym_power_sensor", 2)
power_sensor["id"] = [6, 7]
power_sensor["measured_object"] = [2, 4]
power_sensor["measured_terminal_type"] = [
    MeasuredTerminalType.branch_from, MeasuredTerminalType.load
]
power_sensor["p_measured"] = [1e6, -1e6]
power_sensor["q_measured"] = [1e6, -1e6]
power_sensor["p_sigma"] = [1e4, 1e9]  # trust P for sensor 6
power_sensor["q_sigma"] = [1e9, 1e4]  # trust Q for sensor 7

# all
input_data = {
    "node": node,
    "line": line,
    "sym_load": sym_load,
    "source": source,
    "sym_voltage_sensor": voltage_sensor,
    "sym_power_sensor": power_sensor,
}
